# Praktikum Alternatif Bab 14: End-to-End MLOps: Sistem Rekomendasi Beasiswa Mahasiswa Unindra
### Laboratorium Informatika - Universitas Indraprasta PGRI (Unindra)
**Mata Kuliah:** Machine Learning (Bab 14: Model Deployment & Mini Project)  
**Mahasiswa:** Anggit Arisa Siwi Pratama | **NPM:** `202343570045` | **Kelas:** XC Ekstensi  
**Dosen Pengampu:** Nurfidah Dwitiyanti, M.Si.

---

## Fokus Studi Kasus:
Membangun sistem klasifikasi kelayakan penerima beasiswa internal Unindra berbasis kriteria prestasi akademik (*IPK*), keaktifan organisasi (*poin SKP*), masa studi (*SKS selesai*), dan kondisi ekonomi (*penghasilan orang tua*). Melatih dan membandingkan **3 algoritma klasifikasi**, melakukan tuning hyperparameter, serta mengekspor artefak model untuk antarmuka web.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

print("Pustaka Sains Data & Machine Learning berhasil dimuat.")

In [ ]:
# Inisialisasi Dataset Beasiswa Unindra
np.random.seed(123)
n_mhs = 600

ipk = np.random.uniform(2.5, 4.0, n_mhs)
sks_selesai = np.random.randint(40, 140, n_mhs)
poin_keaktifan = np.random.randint(0, 100, n_mhs)
penghasilan_ortu = np.random.uniform(1_500_000, 15_000_000, n_mhs)

# Syarat beasiswa: IPK tinggi, orang tua prasejahtera, aktif organisasi
skor = (ipk >= 3.5) * 3.0 + (penghasilan_ortu < 5_000_000) * 2.5 + (poin_keaktifan > 50) * 1.5
layak_beasiswa = (skor >= 4.0).astype(int)

df_beasiswa = pd.DataFrame({
    'ipk': np.round(ipk, 2),
    'sks_selesai': sks_selesai,
    'poin_keaktifan': poin_keaktifan,
    'penghasilan_ortu_idr': np.round(penghasilan_ortu, -4),
    'rekomendasi_beasiswa': layak_beasiswa
})

os.makedirs('data', exist_ok=True)
df_beasiswa.to_csv('data/unindra_scholarship_applicants.csv', index=False)
print(f"Dataset Beasiswa Unindra: {df_beasiswa.shape[0]} pelamar.")
print(f"Persentase Direkomendasikan: {df_beasiswa['rekomendasi_beasiswa'].mean()*100:.1f}%")
df_beasiswa.head()

In [ ]:
# Partisi Data & Pelatihan Komparatif 3 Model
X = df_beasiswa.drop(columns=['rekomendasi_beasiswa'])
y = df_beasiswa['rekomendasi_beasiswa']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

alt_models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

hasil_alt = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nama, model in alt_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.0
    
    hasil_alt.append({
        'Model': nama,
        'Akurasi Uji': f'{acc*100:.2f}%',
        'F1-Score': f'{f1*100:.2f}%',
        'ROC-AUC': f'{auc_score:.4f}'
    })

df_hasil_alt = pd.DataFrame(hasil_alt)
print(df_hasil_alt.to_string(index=False))

In [ ]:
# Serialisasi Model Terbaik Beasiswa
os.makedirs('models', exist_ok=True)
best_alt_model = alt_models['Random Forest']
joblib.dump(best_alt_model, 'models/pipeline_beasiswa_unindra.joblib')
print("Model pipeline beasiswa Unindra berhasil disimpan ke models/pipeline_beasiswa_unindra.joblib")

## 4. Kesimpulan & Relevansi Praktikum
Komparasi model menunjukkan bahwa algoritma berbasis pohon ensemble (Random Forest) mampu menangkap aturan kelayakan multi-kriteria secara efektif dengan F1-score tinggi pada data pelamar beasiswa kampus Unindra.